# 02 Cleaning

Use this notebook to clean the raw data, document your transformations, and export a processed file to `data/processed/`.

In [16]:
import pandas as pd
import numpy as np
from pathlib import Path

In [32]:
PROJECT_ROOT = Path('/Users/satyaprakash/nst-dva-capstone-2-project-template')
RAW_PATH = PROJECT_ROOT / 'data/raw/Walmart-Retail-Dataset.csv'
PROCESSED_PATH = PROJECT_ROOT / 'data/processed/cleaned_dataset.csv'

df = pd.read_csv(RAW_PATH, on_bad_lines='skip', low_memory=False)
df.head()
df.isnull().sum()

city                       7
customer_age               0
customer_name             14
customer_segment        2539
discount                   0
order_date                 0
order_id                   0
order_priority          2112
order_quantity             0
product_base_margin        0
product_category           0
product_container       4734
product_name            4745
product_sub_category    4734
profit                  4734
region                  4734
sales                   4734
ship_date               4734
ship_mode               8157
shipping_cost           4734
state                   4734
unit_price              4734
zip_code                4734
dtype: int64

In [18]:
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')


In [19]:
df = df.replace(['\\N', ' ', '', 'NA', 'N/A', 'null'], np.nan)


In [20]:
df = df.drop_duplicates()

In [21]:
num_cols = [
    'customer_age', 'discount', 'order_quantity',
    'product_base_margin', 'profit', 'sales',
    'shipping_cost', 'unit_price', 'zip_code'
]
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [22]:
cat_cols = df.select_dtypes(include='object').columns
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

/var/folders/24/bdgz1hbs05b5qhb7yn_9zch00000gn/T/ipykernel_76399/2841956888.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include='object').columns


In [23]:
for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

In [24]:
df = df[(df['customer_age'] >= 10) & (df['customer_age'] <= 100)]

In [25]:
for col in ['sales', 'profit', 'unit_price', 'shipping_cost']:
    df = df[df[col] >= 0]

In [26]:
df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')
df['ship_date'] = pd.to_datetime(df['ship_date'], errors='coerce')

In [27]:
df['order_year'] = df['order_date'].dt.year
df['order_month'] = df['order_date'].dt.month
df['shipping_days'] = (df['ship_date'] - df['order_date']).dt.days

In [28]:
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(PROCESSED_PATH, index=False)
print(f'Saved cleaned dataset to {PROCESSED_PATH}')

Saved cleaned dataset to /Users/satyaprakash/nst-dva-capstone-2-project-template/data/processed/cleaned_dataset.csv


In [29]:
print(df.isnull().sum())


city                    0
customer_age            0
customer_name           0
customer_segment        0
discount                0
order_date              0
order_id                0
order_priority          0
order_quantity          0
product_base_margin     0
product_category        0
product_container       0
product_name            0
product_sub_category    0
profit                  0
region                  0
sales                   0
ship_date               0
ship_mode               0
shipping_cost           0
state                   0
unit_price              0
zip_code                0
order_year              0
order_month             0
shipping_days           0
dtype: int64
